# Calculate FEM stiffness matrix


In [25]:
# geometry
length_X, length_Y = (600.0, 800.0)
thickness = 25.0

# material properties
young_modulus = 2890.0
poisson_ratio = 0.2
yield_strength = 2.0

# solution parameters
number_of_modes_X = 20
number_of_modes_Y = 20

grid_shape_x = 10
grid_shape_y = 10


In [26]:
from sigmaepsilon.mesh.grid import gridQ4

# we add a margin to avoid point lying on the boundary of the plate, as
# it would lead to a singular compliance matrix
margin = 1.0
gridparams = {
    'size' : (length_X - margin, length_Y - margin),
    'shape' : (grid_shape_x, grid_shape_y),
    'origo' : (0, 0),
    'start' : 0
}
coordsQ4, topoQ4 = gridQ4(**gridparams)
coords2d = coordsQ4[:, :2]
coords2d[:, 0] += margin / 2
coords2d[:, 1] += margin / 2

print(f"Number of nodes: {coords2d.shape[0]}")
print(f"Number of elements: {topoQ4.shape[0]}")
print(f"Min X: {coords2d[:, 0].min()}, Max X: {coords2d[:, 0].max()}")
print(f"Min Y: {coords2d[:, 1].min()}, Max Y: {coords2d[:, 1].max()}")

Number of nodes: 121
Number of elements: 100
Min X: 0.5, Max X: 599.5
Min Y: 0.5, Max Y: 799.5


In [27]:
from time import time
import numpy as np

from sigmaepsilon.math.linalg import ReferenceFrame

from sigmaepsilon.solid.material import MindlinPlateSection as Section
from sigmaepsilon.solid.material import (
    ElasticityTensor,
    LinearElasticMaterial,
    HuberMisesHenckyFailureCriterion_SP,
)
from sigmaepsilon.solid.material.utils import elastic_stiffness_matrix
from sigmaepsilon.solid.fourier import (
    NavierPlate,
    LoadGroup,
    PointLoad,
)

# set up loads
loads = LoadGroup()
num_nodes = coords2d.shape[0]
num_dofs_per_node = 3
total_dofs = num_nodes * num_dofs_per_node
for i in range(num_nodes):
    for j in range(num_dofs_per_node):
        global_dof = i * num_dofs_per_node + j
        load_value = [0.0] * num_dofs_per_node
        load_value[j] = 1.0
        loads[global_dof] = PointLoad(coords2d[i], tuple(load_value))

# setting up hooke's law
hooke = elastic_stiffness_matrix(E=young_modulus, NU=poisson_ratio)
frame = ReferenceFrame(dim=3)
stiffness = ElasticityTensor(hooke, frame=frame, tensorial=False)
failure_model = HuberMisesHenckyFailureCriterion_SP(yield_strength=yield_strength)
material = LinearElasticMaterial(stiffness=stiffness, failure_model=failure_model)

# section stiffness
section = Section(
    layers=[
        Section.Layer(material=material, thickness=thickness),
    ]
)
ABDS_matrix = section.elastic_stiffness_matrix()
bending_stiffness, shear_stiffness = (
    np.ascontiguousarray(ABDS_matrix[:3, :3]),
    np.ascontiguousarray(ABDS_matrix[3:, 3:]),
)

plate = NavierPlate(
    (length_X, length_Y),
    (number_of_modes_X, number_of_modes_Y),
    D=bending_stiffness,
    #S=shear_stiffness,
)

start_time = time()
results = plate.linear_static_analysis(coords2d, loads)
end_time = time()
print(f"Analysis took {end_time - start_time:.4f} seconds.")

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
No implementation of function Function(<built-in function getitem>) found for signature:
 
 >>> getitem(array(float64, 2d, C), Tuple(int64, int64, Literal[int](1)))
 
There are 28 candidate implementations:
   - Of which 26 did not match due to:
   Overload of function 'getitem': File: <numerous>: Line N/A.
     With argument(s): '(array(float64, 2d, C), UniTuple(int64 x 3))':
    No match.
   - Of which 1 did not match due to:
   Overload in function 'GetItemBuffer.generic': File: numba/core/typing/arraydecl.py: Line 211.
     With argument(s): '(array(float64, 2d, C), UniTuple(int64 x 3))':
    Rejected as the implementation raised a specific error:
      NumbaTypeError: cannot index array(float64, 2d, C) with 3 indices: UniTuple(int64 x 3)
  raised from /Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-v2i8RJdg-py3.12/lib/python3.12/site-packages/numba/core/typing/arraydecl.py:133

In [ ]:
plate._linstat_timing_info

{'rhs_assembly_time_seconds': 0.3317708969116211,
 'solution_time_seconds': 0.016087055206298828,
 'postprocessing_time_seconds': 345.42517495155334,
 'result_assembly_time_seconds': 0.08070564270019531}

In [ ]:
compliance_matrix = np.zeros((total_dofs, total_dofs), dtype=float)
for i in range(total_dofs):
    r = results[i].to_pandas()[["UZ", "ROTX", "ROTY"]].values.flatten()
    compliance_matrix[:, i] = r

In [ ]:
start_time = time()
stiffness_matrix = np.linalg.inv(compliance_matrix)
end_time = time()
print(f"Inverting the compliance matrix took {end_time - start_time:.4f} seconds.")
stiffness_matrix.min(), stiffness_matrix.max(), stiffness_matrix.shape

Inverting the compliance matrix took 0.0761 seconds.


(-1.0540112802973202e+25, 5.076282731731496e+24, (1323, 1323))